# 🌳 Deforestation Tracker (Full Auto Pipeline)
Includes Kaggle Download + CNN + ResNet50 + Satellite Imagery + Heatmap

> **Runtime target:** under 3 hours total on a standard Colab GPU.

## 📥 Upload Kaggle API Key

In [ ]:
from google.colab import files
files.upload()  # upload kaggle.json

## 🔐 Setup Kaggle

In [ ]:
import os
os.makedirs('/root/.kaggle', exist_ok=True)
!mv kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json
print('Kaggle credentials set up successfully.')

## 📦 Download Dataset

In [ ]:
!kaggle competitions download -c dsc6232-rwanda-summer2020-hw2
!unzip -q dsc6232-rwanda-summer2020-hw2.zip -d data
print('Dataset downloaded and extracted.')

## 🔧 Convert Dataset

In [ ]:
import os, shutil, pandas as pd

os.makedirs('dataset/forest', exist_ok=True)
os.makedirs('dataset/deforested', exist_ok=True)

# FIX: Explicit error messages instead of bare except
try:
    df = pd.read_csv('data/train.csv')
    print(f'Loaded {len(df)} rows from train.csv')
    print('Columns found:', df.columns.tolist())

    skipped = 0
    copied = 0
    for _, row in df.iterrows():
        img_path = f"data/{row['image_id']}.png"
        if os.path.exists(img_path):
            dest = 'dataset/deforested' if row['label'] == 1 else 'dataset/forest'
            shutil.copy(img_path, dest)
            copied += 1
        else:
            skipped += 1

    print(f'Copied: {copied} images | Skipped (not found): {skipped}')

except FileNotFoundError as e:
    print(f'ERROR: {e}')
    print('Check that train.csv exists in the data/ folder and image_id column is correct.')
except KeyError as e:
    print(f'ERROR: Column {e} not found in CSV.')
    print('Available columns:', df.columns.tolist())

## 🤖 Train Models

> **Time budget:** 3 epochs each is intentional — enough for meaningful transfer learning convergence without exceeding the 3-hour total runtime on a T4 GPU.

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_dir = 'dataset'

datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train_data = datagen.flow_from_directory(
    train_dir, target_size=(224, 224), batch_size=32,
    class_mode='binary', subset='training'
)
val_data = datagen.flow_from_directory(
    train_dir, target_size=(224, 224), batch_size=32,
    class_mode='binary', subset='validation'
)

print('Class indices:', train_data.class_indices)

In [ ]:
from tensorflow.keras import layers, models

cnn = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])
cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print('Training CNN (3 epochs)...')
cnn_history = cnn.fit(train_data, validation_data=val_data, epochs=3)

# FIX: Save model so it's not lost if session ends
cnn.save('cnn_deforestation.h5')
print('CNN saved to cnn_deforestation.h5')

In [ ]:
from tensorflow.keras.applications import ResNet50

base = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
for layer in base.layers:
    layer.trainable = False

resnet = models.Sequential([
    base,
    layers.GlobalAveragePooling2D(),
    layers.Dense(1, activation='sigmoid')
])
resnet.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print('Training ResNet50 (3 epochs)...')
resnet_history = resnet.fit(train_data, validation_data=val_data, epochs=3)

# FIX: Save model so it's not lost if session ends
resnet.save('resnet_deforestation.h5')
print('ResNet50 saved to resnet_deforestation.h5')

## 📊 Compare Models

In [ ]:
import matplotlib.pyplot as plt

cnn_acc = cnn.evaluate(val_data, verbose=0)[1]
resnet_acc = resnet.evaluate(val_data, verbose=0)[1]

acc = {'CNN': cnn_acc, 'ResNet50': resnet_acc}

plt.figure(figsize=(6, 4))
bars = plt.bar(acc.keys(), acc.values(), color=['steelblue', 'seagreen'])
plt.ylim(0, 1)
plt.ylabel('Validation Accuracy')
plt.title('Model Comparison')
for bar, val in zip(bars, acc.values()):
    plt.text(bar.get_x() + bar.get_width() / 2, val + 0.01, f'{val:.2%}', ha='center')
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=100)
plt.show()

print(f'CNN Accuracy:    {cnn_acc:.2%}')
print(f'ResNet Accuracy: {resnet_acc:.2%}')

## 🛰️ Fetch Satellite Image

> Uses Google Earth Engine (GEE) to pull a **real Sentinel-2 multispectral image** over the Rwanda region.
>
> **Why satellite imagery?** The Kaggle dataset trains the model on labelled 224×224 patches. But to actually *use* the model on new, unseen land — we need a live satellite image. GEE provides free access to Sentinel-2 imagery (10m/pixel resolution, updated every 5 days). We slice that image into patches, run the model on each one, and build a heatmap showing where deforestation is predicted.

In [ ]:
import ee
import requests

ee.Authenticate()
ee.Initialize()

# Rwanda region bounding box
region = ee.Geometry.Rectangle([29.5, -2.5, 30.5, -1.5])

# FIX: Added filterDate to avoid cloudy/stale imagery — using last 6 months
img_collection = (
    ee.ImageCollection('COPERNICUS/S2')
    .filterBounds(region)
    .filterDate('2024-06-01', '2024-12-31')   # recent, low-cloud window
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))  # skip cloudy images
    .sort('CLOUDY_PIXEL_PERCENTAGE')           # least cloudy first
)

satellite_img = img_collection.first()
print('Selected image:', satellite_img.get('system:index').getInfo())

url = satellite_img.getThumbURL({
    'region': region,
    'dimensions': 512,
    'bands': ['B4', 'B3', 'B2'],   # RGB true colour
    'min': 0,
    'max': 3000
})

response = requests.get(url)
response.raise_for_status()  # raises error if download failed

# FIX: Use context manager so file handle is properly closed
with open('sat.png', 'wb') as f:
    f.write(response.content)

print('Satellite image saved to sat.png')

## 🔧 Patch Split + Predict

In [ ]:
import cv2
import numpy as np
import os

os.makedirs('patches', exist_ok=True)

# FIX: Renamed to satellite_img_cv2 so it doesn't conflict with the GEE 'satellite_img' above
satellite_img_cv2 = cv2.imread('sat.png')
if satellite_img_cv2 is None:
    raise FileNotFoundError('sat.png not found — make sure the satellite download cell ran successfully.')

h, w = satellite_img_cv2.shape[:2]
print(f'Satellite image size: {w}x{h} pixels')

patch_paths = []
patch_coords = []  # track (i, j) for heatmap reconstruction

for i in range(0, h, 224):
    for j in range(0, w, 224):
        patch = satellite_img_cv2[i:i+224, j:j+224]

        # FIX: Check BOTH dimensions to avoid shape mismatch on edge patches
        if patch.shape[0] == 224 and patch.shape[1] == 224:
            path = f'patches/{len(patch_paths)}.png'
            cv2.imwrite(path, patch)
            patch_paths.append(path)
            patch_coords.append((i, j))

print(f'Total valid patches: {len(patch_paths)}')

# Run inference with ResNet (best model)
results = []
for path in patch_paths:
    patch_img = cv2.imread(path)
    patch_img = cv2.resize(patch_img, (224, 224)) / 255.0
    patch_img = np.reshape(patch_img, (1, 224, 224, 3))
    pred = resnet.predict(patch_img, verbose=0)[0][0]
    results.append(1 if pred > 0.5 else 0)

deforested_count = sum(results)
print(f'Patches predicted deforested: {deforested_count}/{len(results)} ({deforested_count/len(results):.1%})')

## 🔥 Heatmap

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# FIX: Use satellite_img_cv2 (the cv2 array) consistently — no variable name conflict
h, w = satellite_img_cv2.shape[:2]
heat = np.zeros((h, w))

# FIX: Use patch_coords for precise placement instead of re-computing loop indices
for (i, j), label in zip(patch_coords, results):
    heat[i:i+224, j:j+224] = label

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Show original satellite image alongside heatmap
rgb = cv2.cvtColor(satellite_img_cv2, cv2.COLOR_BGR2RGB)
axes[0].imshow(rgb)
axes[0].set_title('Satellite Image (Rwanda Region)')
axes[0].axis('off')

im = axes[1].imshow(heat, cmap='hot', vmin=0, vmax=1)
axes[1].set_title('Deforestation Heatmap (Red = Deforested)')
axes[1].axis('off')
plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.savefig('deforestation_heatmap.png', dpi=150)
plt.show()

print('Heatmap saved to deforestation_heatmap.png')